# AI Math Assistant — walkthrough

Tool calling with LangChain: `@tool` functions + `create_agent`.
Runs **offline by default** (rule-based stand-in model, no API key). Set `PROVIDER = "openai"` or `"watsonx"` to use a real LLM.

In [1]:
PROVIDER = "offline"  # "openai" | "watsonx" | "offline"

from dotenv import load_dotenv
load_dotenv()

from math_assistant import (MATH_TOOLS, power_tool, add_numbers, divide_numbers,
                            build_agent, get_llm, ask, final_answer, tool_calls)
from math_assistant.cli import print_trace

## 1. Tools are plain functions with a name, a description and a schema
The LLM only sees the name, the docstring and the argument schema — that's how it decides which tool to call.

In [2]:
for t in MATH_TOOLS:
    print(f"{t.name:18} args={list(t.args)}  | {t.description.splitlines()[0]}")

add_numbers        args=['inputs']  | Add all numbers found in the input string, e.g. "10, 20, 30" -> 60.
subtract_numbers   args=['inputs']  | Subtract sequentially: first number minus every following number.
multiply_numbers   args=['inputs']  | Multiply all numbers found in the input string, e.g. "2, 3, 4" -> 24.
divide_numbers     args=['inputs']  | Divide the first number by each following number, e.g. "100, 5, 2" -> 10.
power_tool         args=['input_text']  | Calculate a power: base (x) raised to exponent (y), i.e. x^y.


## 2. Test tools directly before giving them to an agent

In [3]:
print(add_numbers.invoke("10 20 30 a b"))
print(divide_numbers.invoke("100, 5, 2"))
print(divide_numbers.invoke("10, 0"))          # errors are returned, not raised
print(power_tool.invoke("2 to the power of 8"))

{'result': 60.0}
{'result': 10.0}
{'result': 'Cannot divide by zero.'}
{'result': 256.0}


## 3. Build the agent and ask a question

In [4]:
agent = build_agent(get_llm(PROVIDER))
response = ask(agent, "What is 25 divided by 4?")
final_answer(response)

'The result is 6.25.'

## 4. Look inside the loop: Human → AI tool call → Tool result → AI answer

In [5]:
print_trace(ask(agent, "Add 25 and 15, then multiply by 2"))

[1] Human      Add 25 and 15, then multiply by 2
[2] AI → tool  add_numbers({'inputs': '25, 15'})
[3] Tool       add_numbers → {"result": 40.0}
[4] AI → tool  multiply_numbers({'inputs': '40, 2'})
[5] Tool       multiply_numbers → {"result": 80.0}
[6] AI answer  The result is 80.


## 5. Evaluate: assert on the tool result, not the wording

In [6]:
cases = [("Subtract 100, 20, and 10.", 70), ("Multiply 2, 3, and 4.", 24),
         ("Divide 100 by 5 and then by 2.", 10), ("Subtract 50 from 20.", -30),
         ("Calculate 5 to the power of 2.", 25), ("What is 12 squared?", 144)]

passed = 0
for query, expected in cases:
    calls = tool_calls(ask(agent, query))
    result = calls[-1]["result"] if calls else None
    ok = result == expected
    passed += ok
    print(f"{'✅' if ok else '❌'} {query:35} -> {result}")
print(f"\n{passed}/{len(cases)} passed")

✅ Subtract 100, 20, and 10.           -> 70.0
✅ Multiply 2, 3, and 4.               -> 24.0
✅ Divide 100 by 5 and then by 2.      -> 10.0
✅ Subtract 50 from 20.                -> -30.0
✅ Calculate 5 to the power of 2.      -> 25.0
✅ What is 12 squared?                 -> 144.0

6/6 passed
